# 01 — Exploratory Data Analysis: BraTS 2020

**Goal:** Understand the raw data before writing a single model line.

Sections:
1. Dataset inventory
2. Raw NIfTI inspection (volumes + voxel stats)
3. Prepared 2D slice inspection
4. Class distribution
5. Modality statistics
6. Sample grid visualisation

In [ ]:
import sys
from pathlib import Path

# Allow imports from project root
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Kaggle path (change if running locally)
BRATS_RAW = Path('/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData')
BRATS_2D  = Path('/kaggle/working/brats2020_2d')   # output of prepare_data.py

print('ROOT:', ROOT)
print('Raw data exists:', BRATS_RAW.exists())
print('Prepared 2D exists:', BRATS_2D.exists())

In [ ]:
import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Dataset Inventory

In [ ]:
patient_dirs = sorted(BRATS_RAW.glob('BraTS20_Training_*'))
print(f'Total training patients: {len(patient_dirs)}')
print('First 5:', [d.name for d in patient_dirs[:5]])

# Confirm expected files for one patient
sample_patient = patient_dirs[0]
modality_files = list(sample_patient.glob('*.nii'))
print(f'\nFiles in {sample_patient.name}:')
for f in sorted(modality_files):
    print(f'  {f.name}')

## 2. Raw NIfTI Inspection

In [ ]:
def load_volume(patient_dir: Path, modality: str) -> np.ndarray:
    """Load a single NIfTI modality for a patient."""
    matches = list(patient_dir.glob(f'*_{modality}.nii'))
    if not matches:
        matches = list(patient_dir.glob(f'*_{modality}.nii.gz'))
    return nib.load(matches[0]).get_fdata(dtype=np.float32)

p = patient_dirs[0]
flair = load_volume(p, 'flair')
t1    = load_volume(p, 't1')
t1ce  = load_volume(p, 't1ce')
t2    = load_volume(p, 't2')
seg   = load_volume(p, 'seg').astype(np.uint8)

print(f'Volume shape: {flair.shape}   (H × W × D)')  # expected: (240, 240, 155)
print(f'Seg labels present: {np.unique(seg)}')

In [ ]:
# Voxel-level stats across modalities
for name, vol in [('FLAIR', flair), ('T1', t1), ('T1ce', t1ce), ('T2', t2)]:
    brain = vol[vol > 0]
    print(f'{name:6s}  min={brain.min():.1f}  max={brain.max():.1f}  '
          f'mean={brain.mean():.1f}  std={brain.std():.1f}  '
          f'brain_voxels={len(brain):,}')

In [ ]:
# 3-plane view: axial, coronal, sagittal for FLAIR + segmentation
cx, cy, cz = [s // 2 for s in flair.shape]  # centre voxel

label_colors = np.array([
    [0,   0,   0,   0  ],  # background — transparent
    [230, 51,  51,  180],  # NCR/NET — red
    [51,  204, 51,  180],  # edema — green
    [230, 230, 51,  180],  # enhancing — yellow
], dtype=np.uint8)

def seg_rgba(seg_slice: np.ndarray) -> np.ndarray:
    # BraTS native labels: 0,1,2,4 → remap 4→3
    s = seg_slice.copy()
    s[s == 4] = 3
    return label_colors[s]

planes = [
    ('Axial',    flair[:, :, cz],    seg[:, :, cz]),
    ('Coronal',  flair[:, cy, :],    seg[:, cy, :]),
    ('Sagittal', flair[cx, :, :],    seg[cx, :, :]),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (title, img, s) in zip(axes, planes):
    ax.imshow(img.T, cmap='gray', origin='lower')
    ax.imshow(seg_rgba(s.T), origin='lower')
    ax.set_title(title)
    ax.axis('off')
fig.suptitle(f'Patient: {p.name} — FLAIR + segmentation overlay', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# All 4 modalities at the centre axial slice
modalities = [('FLAIR', flair), ('T1', t1), ('T1ce', t1ce), ('T2', t2)]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, vol) in zip(axes, modalities):
    ax.imshow(vol[:, :, cz].T, cmap='gray', origin='lower')
    ax.imshow(seg_rgba(seg[:, :, cz].T), origin='lower')
    ax.set_title(name)
    ax.axis('off')
fig.suptitle(f'{p.name} — axial slice {cz} — all modalities', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Intensity histograms (brain voxels only)
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, (name, vol) in zip(axes, modalities):
    brain = vol[vol > 0].ravel()
    ax.hist(brain, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{name} (n={len(brain):,})')
    ax.set_xlabel('Intensity')
    ax.set_ylabel('Count')
fig.suptitle('Intensity distributions — brain voxels only', fontsize=11)
plt.tight_layout()
plt.show()

## 3. Prepared 2D Slices

In [ ]:
with open(BRATS_2D / 'metadata.json') as f:
    metadata = json.load(f)

slices = metadata['slices']
patients = sorted({s['patient'] for s in slices})

print(f'Total 2D slices: {len(slices):,}')
print(f'Patients: {len(patients)}')
print(f'Avg slices per patient: {len(slices)/len(patients):.1f}')

# Slices per patient distribution
from collections import Counter
spp = Counter(s['patient'] for s in slices)
vals = list(spp.values())
print(f'Slices/patient — min: {min(vals)}  max: {max(vals)}  median: {np.median(vals):.0f}')

In [ ]:
# Slices per patient histogram
plt.figure(figsize=(8, 3))
plt.hist(vals, bins=20, color='steelblue', edgecolor='white')
plt.xlabel('Slices per patient')
plt.ylabel('Count')
plt.title('Distribution of 2D slice counts per patient')
plt.tight_layout()
plt.show()

In [ ]:
# Load a few prepared slices and check shapes
sample_patient_id = patients[0]
patient_slices = [s for s in slices if s['patient'] == sample_patient_id]
mid = patient_slices[len(patient_slices) // 2]

img = np.load(BRATS_2D / mid['patient'] / f"slice_{mid['slice']:03d}_image.npy")
msk = np.load(BRATS_2D / mid['patient'] / f"slice_{mid['slice']:03d}_mask.npy")

print(f'image shape: {img.shape}  dtype: {img.dtype}')
print(f'mask  shape: {msk.shape}  dtype: {msk.dtype}  unique: {np.unique(msk)}')
print(f'Per-channel stats after z-score:')
for i, name in enumerate(['FLAIR', 'T1', 'T1ce', 'T2']):
    ch = img[i]
    brain = ch[ch != 0]
    print(f'  {name}: mean={brain.mean():.3f}  std={brain.std():.3f}')

## 4. Class Distribution

In [ ]:
# Count pixels per class across ALL prepared slices
# This is slow — runs through every .npy mask file
from tqdm.auto import tqdm

class_counts = np.zeros(4, dtype=np.int64)
for s in tqdm(slices, desc='counting pixels'):
    msk_path = BRATS_2D / s['patient'] / f"slice_{s['slice']:03d}_mask.npy"
    m = np.load(msk_path)
    for c in range(4):
        class_counts[c] += (m == c).sum()

total = class_counts.sum()
class_names = ['background', 'NCR/NET', 'edema', 'enhancing']
print('Class pixel counts and percentages:')
for i, (name, cnt) in enumerate(zip(class_names, class_counts)):
    print(f'  [{i}] {name:15s}: {cnt:>12,}  ({100*cnt/total:.2f}%)')

In [ ]:
# Class imbalance bar chart (excluding background to see tumor classes)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#555555', '#e83333', '#33cc33', '#e8e833']

axes[0].bar(class_names, class_counts, color=colors)
axes[0].set_title('All classes (absolute pixel counts)')
axes[0].set_ylabel('Pixels')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(class_names[1:], class_counts[1:], color=colors[1:])
axes[1].set_title('Tumor classes only (background excluded)')
axes[1].set_ylabel('Pixels')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# Imbalance ratio
tumor_counts = class_counts[1:]
print(f'\nTumor class imbalance (max/min): {tumor_counts.max()/tumor_counts.min():.1f}x')
print(f'Background vs all tumor: {class_counts[0]/class_counts[1:].sum():.1f}x more background')

## 5. Modality Statistics Across Patients

In [ ]:
# Per-modality stats: sample 20 patients for speed
rng = np.random.default_rng(42)
sample_pids = rng.choice(patients, size=min(20, len(patients)), replace=False)

modality_stats = {name: {'means': [], 'stds': []} for name in ['FLAIR', 'T1', 'T1ce', 'T2']}

for pid in tqdm(sample_pids, desc='patient stats'):
    pid_slices = [s for s in slices if s['patient'] == pid]
    all_voxels = {name: [] for name in modality_stats}
    for s in pid_slices:
        img = np.load(BRATS_2D / s['patient'] / f"slice_{s['slice']:03d}_image.npy")
        for i, name in enumerate(['FLAIR', 'T1', 'T1ce', 'T2']):
            ch = img[i]
            brain = ch[ch != 0]
            if len(brain):
                all_voxels[name].append(brain)
    for name in modality_stats:
        if all_voxels[name]:
            combined = np.concatenate(all_voxels[name])
            modality_stats[name]['means'].append(combined.mean())
            modality_stats[name]['stds'].append(combined.std())

print('Per-modality stats (mean ± std across sampled patients):')
for name, stats in modality_stats.items():
    means = np.array(stats['means'])
    stds  = np.array(stats['stds'])
    print(f'  {name:6s}  mean={means.mean():.3f}±{means.std():.3f}  std={stds.mean():.3f}±{stds.std():.3f}')

In [ ]:
# Boxplot of per-patient means
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names = list(modality_stats.keys())

means_data = [modality_stats[n]['means'] for n in names]
stds_data  = [modality_stats[n]['stds']  for n in names]

axes[0].boxplot(means_data, labels=names)
axes[0].set_title('Per-patient brain-voxel mean (after z-score)')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Mean intensity')

axes[1].boxplot(stds_data, labels=names)
axes[1].set_title('Per-patient brain-voxel std (after z-score)')
axes[1].axhline(1, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Std intensity')

plt.tight_layout()
plt.show()

## 6. Sample Grid Visualisation

In [ ]:
from src.data.brats import BraTSDataset
from src.data.splits import load_split
from src.data.visualize import plot_modalities, plot_sample_grid, MASK_CMAP

split = load_split(ROOT / 'configs/splits/default.json')
train_ds = BraTSDataset(BRATS_2D, patient_ids=split['train'])
print(f'Train slices: {len(train_ds):,}  |  Patients: {len(train_ds.patient_ids)}')

In [ ]:
# Single sample: 4 modalities side-by-side with mask overlay
img, msk = train_ds[len(train_ds) // 2]
fig = plot_modalities(img.numpy(), msk.numpy(), title='Middle training sample — all modalities')
plt.show()

In [ ]:
# Grid of 8 random samples
fig = plot_sample_grid(train_ds, n=8, seed=42)
plt.show()

In [ ]:
# Find and display a tumor-heavy slice
tumor_fracs = []
check_n = min(500, len(train_ds))
for i in range(check_n):
    _, msk = train_ds[i]
    frac = (msk > 0).float().mean().item()
    tumor_fracs.append((frac, i))

tumor_fracs.sort(reverse=True)
top_idx = tumor_fracs[0][1]
img, msk = train_ds[top_idx]
fig = plot_modalities(img.numpy(), msk.numpy(),
                      title=f'Most tumor-dense slice (index {top_idx}, tumor fraction={tumor_fracs[0][0]:.2%})')
plt.show()

# Distribution of tumor fraction across checked slices
fracs = [f for f, _ in tumor_fracs]
plt.figure(figsize=(7, 3))
plt.hist(fracs, bins=40, color='steelblue', edgecolor='none')
plt.xlabel('Tumor pixel fraction per slice')
plt.ylabel('Count')
plt.title(f'Tumor fraction distribution (first {check_n} training slices)')
plt.tight_layout()
plt.show()

## Summary

Key findings from this EDA:

| Finding | Implication |
|---|---|
| Background ≈ 98%+ of pixels | Need Dice loss (not cross-entropy alone) to handle imbalance |
| Enhancing tumor is rarest class | May need class weights or oversampling |
| FLAIR shows edema best | Important to keep all 4 modalities |
| T1ce highlights enhancing tumor | Critical modality for highest-value class |
| Z-score shifts means to ~0 | Normalization working correctly |
| Patient slice counts vary | Patient-level split was essential |
